# Language-Model Adapter — Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_artifact_inference_colab.ipynb)

Someone hands you a 36 MB ZIP and says "this is the fine-tuned model". It is not a model; it is a **PEFT adapter**: a few million low-rank delta weights that only mean something on top of one specific base model at one specific revision. Before you run it you want to know that nothing in the archive was altered in transit, that it names the base model it was trained on, and that it will not execute code from anywhere.

This notebook consumes the bundle produced by the companion [fine-tuning tutorial](language_model_finetuning_colab.ipynb) and shows the checks and the load path you would use in a real service.

**By the end of this notebook you will be able to:**
- **Verify** an adapter bundle against its own manifest: safe paths, no symlinks, every file's size and SHA-256, no extra files.
- **Resolve** the exact base model from the bundle's provenance and load it, online from a pinned Hub revision or offline from a verified snapshot.
- **Attach** the adapter with `peft` and show, prompt by prompt, what it changes relative to the base model.
- **Choose decoding settings** deliberately: greedy for reproducible checks, sampling for real use, and why thinking-mode models need the latter.

## Prerequisites

- The `dimer-language-model-adapter.zip` from the fine-tuning notebook (or any bundle in the same format: `artifact-manifest.json` + `provenance.json` + `adapter_model.safetensors` + `tokenizer/`).
- A Colab GPU runtime (*Runtime ▸ Change runtime type ▸ T4 GPU*). The default base model is 1.4 GB and loads in 4-bit; the whole notebook takes about two minutes after the upload.
- An `HF_TOKEN` in Colab Secrets only if the bundle's base model is gated (Llama 3.2).

> upload adapter ZIP → verify manifest, hashes and provenance → resolve the exact base model → attach the adapter → compare base vs adapted → generate


## 1. Install and verify the artifact

The install line pins the same library versions the adapter was trained with; adapter serialization and chat-template rendering are not stable across major releases.

Then the uploaded ZIP is treated as untrusted input, which it is. Before a single file is read as data, the code checks:
- **paths**: every member is a relative path without `..`, without a leading `/`, without backslashes, and resolves inside the extraction folder;
- **symlinks**: refused outright (a symlink inside an archive can point anywhere on the machine);
- **size**: the archive may not expand past 512 MiB;
- **manifest**: exactly one `artifact-manifest.json`, whose every entry matches a file's size and SHA-256, with no files on disk that the manifest does not list.

Optionally paste the whole-archive SHA-256 the sender gave you into `EXPECTED_ARTIFACT_ZIP_SHA256`; then even a substituted manifest is caught.


In [ ]:
%pip -q install transformers==4.57.1 tokenizers==0.22.1 huggingface-hub==0.36.0 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.49.0 safetensors==0.8.0


Run the next cell and pick the adapter ZIP when the upload dialog appears.


In [ ]:
import hashlib
import json
import re
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

import pandas as pd
import torch
from huggingface_hub import HfApi
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Who wrote this notebook; recorded for provenance, not a sign-off.
AI_PROVENANCE = {
    "builders": [
        {"provider": "OpenAI", "product": "ChatGPT", "model": "GPT-5.6 Sol High", "role": "Builder"},
        {"provider": "Anthropic", "product": "Claude Code", "model": "Claude Fable 5.1", "role": "Reviewer and content revision"},
    ]
}
if not torch.cuda.is_available():
    raise RuntimeError("No GPU visible. In Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.")

EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}

from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one adapter ZIP")
archive_path = Path("/content") / Path(next(iter(uploaded))).name
archive_path.write_bytes(next(iter(uploaded.values())))


def sha256_of_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha = sha256_of_file(archive_path)
if EXPECTED_ARTIFACT_ZIP_SHA256 and archive_sha.lower() != EXPECTED_ARTIFACT_ZIP_SHA256.strip().lower():
    raise ValueError("Whole-ZIP SHA-256 mismatch: this is not the archive you were told to expect")
print(f"archive {archive_path.name}: {archive_path.stat().st_size / 1024**2:.1f} MB, sha256 {archive_sha}")


def manifest_member_path(root, member_name):
    """Resolve a member name under root, refusing anything that could escape it."""
    if "\\" in member_name:
        raise ValueError(f"Unsafe artifact path (backslash): {member_name!r}")
    relative = PurePosixPath(member_name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe artifact path: {member_name!r}")
    root = Path(root).resolve()
    target = (root / Path(*relative.parts)).resolve()
    if target != root and root not in target.parents:
        raise ValueError(f"Artifact path escapes the extraction root: {member_name!r}")
    return target


def extract_zip_safely(zip_path, root, size_limit_bytes):
    root = Path(root).resolve()
    shutil.rmtree(root, ignore_errors=True)
    root.mkdir(parents=True)
    expanded = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            if stat.S_ISLNK((info.external_attr >> 16) & 0xFFFF):
                raise ValueError("Symlinks are not allowed in the archive")
            expanded += info.file_size
            if expanded > size_limit_bytes:
                raise ValueError("Archive expands beyond the allowed size")
            target = manifest_member_path(root, info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, open(target, "wb") as destination:
                shutil.copyfileobj(source, destination)
    return root


extraction_root = extract_zip_safely(archive_path, "/content/dimer-language-model-artifact", size_limit_bytes=512 * 1024**2)
manifests = list(extraction_root.rglob("artifact-manifest.json"))
if len(manifests) != 1:
    raise ValueError("Expected exactly one artifact-manifest.json in the archive")
artifact_root = manifests[0].parent
manifest = json.loads(manifests[0].read_text())
if manifest.get("format") != "peft_adapter" or manifest.get("formatVersion") != 1:
    raise ValueError("Unsupported artifact format")

listed, listed_bytes = set(), 0
for record in manifest.get("files", []):
    file_path = manifest_member_path(artifact_root, record["path"])
    if not file_path.is_file() or file_path.stat().st_size != record["bytes"] or sha256_of_file(file_path) != record["sha256"]:
        raise ValueError(f"SHA-256 mismatch: {record['path']}")
    listed.add(record["path"])
    listed_bytes += record["bytes"]
on_disk = {p.relative_to(artifact_root).as_posix() for p in artifact_root.rglob("*") if p.is_file() and p.name != "artifact-manifest.json"}
if on_disk != listed or listed_bytes != manifest.get("totalBytes"):
    raise ValueError("Manifest/file-set mismatch: files were added, removed, or the total size differs")
print(f"Adapter manifest verified: {len(listed)} files, {listed_bytes / 1024**2:.1f} MB")


## 2. Resolve the base model from provenance

`provenance.json` says which base model the deltas belong to. Two fields are load-bearing:

- **`baseModelRevision`** must be a 40-character commit SHA. A branch name like `main` is rejected: the Hub can move it, and an adapter attached to weights it was not trained on produces confident nonsense rather than an error.
- **`trustRemoteCode`** must be `false`. Nothing here will ever import Python from a model repository.

If the base model is gated, the notebook reads `HF_TOKEN` from Colab Secrets and confirms with the Hub that the pinned revision exists before any download starts. The token stays in memory.


In [ ]:
provenance = json.loads((artifact_root / "provenance.json").read_text())
if provenance.get("trustRemoteCode") is not False:
    raise ValueError("trustRemoteCode must be false")
base_model_id = provenance.get("baseModel")
base_revision = provenance.get("baseModelRevision")
if not re.fullmatch(r"[0-9a-f]{40}", base_revision or ""):
    raise ValueError("Invalid baseModelRevision provenance: expected a 40-character commit SHA, not a branch or tag")

BASE_MODEL_SOURCE = "Pinned Hugging Face" # @param ["Pinned Hugging Face","DIMER ZIP"]
DIMER_BASE_ZIP_PATH = "/content/dimer-base-model.zip" # @param {type:"string"}

requires_token = bool(provenance.get("requiresHfToken"))
dimer_zip_allowed = bool(provenance.get("dimerZipAllowed", True))
model_key = provenance.get("modelKey")
if BASE_MODEL_SOURCE == "DIMER ZIP" and not dimer_zip_allowed:
    raise RuntimeError("This artifact's base model may not be redistributed as an offline ZIP")

HF_TOKEN = None
if BASE_MODEL_SOURCE == "Pinned Hugging Face" and requires_token:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable Notebook access") from exc
    hub_info = HfApi(token=HF_TOKEN).model_info(base_model_id, revision=base_revision)
    if hub_info.sha != base_revision:
        raise RuntimeError("Pinned revision mismatch")
    print("Hugging Face credential and pinned-revision preflight passed")

training = provenance.get("training", {})
print(f"base model: {base_model_id} @ {base_revision[:12]} | license {provenance.get('baseModelLicense')}")
print(f"trained with: {training} | dataset: {provenance.get('dataset')}")


## 3. Acquire the base model

**`Pinned Hugging Face`** downloads exactly `baseModelRevision`. **`DIMER ZIP`** uses an offline snapshot instead (built with the repository's `scripts/fetch_weights.py --zip`); the snapshot's own manifest must name the *same* model and revision as the adapter's provenance, and every file must match its recorded hash, before it is loaded with `local_files_only=True`.


In [ ]:
BASE_MODEL_LOAD_REF = base_model_id
if BASE_MODEL_SOURCE == "DIMER ZIP":
    snapshot_root = extract_zip_safely(DIMER_BASE_ZIP_PATH, "/content/dimer-inference-base", size_limit_bytes=20 * 1024**3)
    snapshot_manifests = list(snapshot_root.rglob("dimer-base-manifest.json"))
    if len(snapshot_manifests) != 1:
        raise ValueError("Expected exactly one dimer-base-manifest.json in the base-model ZIP")
    model_root = snapshot_manifests[0].parent
    base_manifest = json.loads(snapshot_manifests[0].read_text())
    identity = (base_manifest.get("modelKey"), base_manifest.get("modelId"), base_manifest.get("revision"))
    if base_manifest.get("format") != "dimer_hf_snapshot" or identity != (model_key, base_model_id, base_revision):
        raise ValueError("The base-model ZIP describes a different model or revision than the adapter's provenance")
    for record in base_manifest.get("files", []):
        file_path = manifest_member_path(model_root, record["path"])
        if not file_path.is_file() or file_path.stat().st_size != record["bytes"] or sha256_of_file(file_path) != record["sha256"]:
            raise ValueError(f"SHA-256 mismatch: {record['path']}")
    BASE_MODEL_LOAD_REF = model_root
    print("DIMER base-model package verified")


## 4. Attach the adapter and compare it with the base model

The base model loads in 4-bit `nf4` (the same way it was trained) with memory-efficient attention. The tokenizer comes **from the bundle**, not from the Hub, so prompts render with exactly the chat template the adapter saw in training. `PeftModel.from_pretrained(..., is_trainable=False)` attaches the deltas in eval mode.

The first comparison uses **greedy decoding** (`do_sample=False`) and **non-thinking mode** so that the result is deterministic and the only difference between the two columns is the adapter itself: `model.disable_adapter()` switches it off for the base answer. Expect the answers to be similar in substance and to differ in language, tone or format; that is what a small adapter trained on ~100 rows changes.


In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(artifact_root / "tokenizer", local_files_only=True, trust_remote_code=False)
load_kwargs = {"trust_remote_code": False, "dtype": compute_dtype, "quantization_config": quant_config, "device_map": {"": 0}, "attn_implementation": "sdpa"}
if BASE_MODEL_SOURCE == "Pinned Hugging Face":
    load_kwargs["revision"] = base_revision
    if HF_TOKEN:
        load_kwargs["token"] = HF_TOKEN
else:
    load_kwargs["local_files_only"] = True

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_LOAD_REF, **load_kwargs)
model = PeftModel.from_pretrained(base, artifact_root, is_trainable=False)
model.eval()
print(f"loaded {base_model_id} in 4-bit + adapter | GPU memory {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")


def render_prompt(prompt):
    # enable_thinking=False asks Qwen3-style templates for a direct answer; other templates ignore it.
    return tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=False)


def reply(prompt, max_new_tokens=96, **decoding):
    """Generate a reply. Pass do_sample/temperature/top_p/top_k in `decoding`; default is greedy."""
    inputs = tokenizer(render_prompt(prompt), return_tensors="pt", add_special_tokens=False).to("cuda")
    settings = {"do_sample": False, **decoding}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id, **settings)
    return tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def base_and_adapted(prompt):
    with model.disable_adapter():
        base_answer = reply(prompt)
    return base_answer, reply(prompt)


PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.",
]
pairs = [base_and_adapted(prompt) for prompt in PROMPTS]
display(pd.DataFrame({"prompt": PROMPTS, "base (adapter off)": [b for b, _ in pairs], "adapted": [a for _, a in pairs]}))
print("✓ Artifact verified, base resolved, adapter attached, generation complete")


## 5. Decoding settings for real use

Greedy decoding is right for a reproducible check and wrong for a product: small models under greedy decoding fall into repetition loops ("…magandang magandang magandang…"), and thinking-mode models loop in their reasoning trace. For actual use, sample:

| Situation | Settings |
|---|---|
| Direct answers (non-thinking mode, what this adapter was trained for) | `do_sample=True, temperature=0.7, top_p=0.8, top_k=20` |
| Thinking mode (`<think>…</think>` traces on Qwen3-style models) | `do_sample=True, temperature=0.6, top_p=0.95, top_k=20`, and render prompts *without* `enable_thinking=False` |
| Reproducible smoke tests and regression checks | `do_sample=False` (greedy), as in Section 4 |

The cell below re-asks the first prompt with the direct-answer settings. Run it a few times: each answer will differ, and none should loop.


In [ ]:
SAMPLING = {"do_sample": True, "temperature": 0.7, "top_p": 0.8, "top_k": 20}
for attempt in range(2):
    print(f"sampled answer {attempt + 1}: {reply(PROMPTS[0], **SAMPLING)}\n")


## Recap and next steps

You verified an untrusted archive before reading it, tied it to an immutable base-model revision, attached it, and saw what the adapter changes with the adapter switched off and on. Against the objectives:
- *Verify*: Section 1 (safe extraction, manifest, hashes, optional whole-archive digest).
- *Resolve*: Sections 2–3 (40-character SHA, `trustRemoteCode`, online or offline base).
- *Attach and compare*: Section 4 (`PeftModel.from_pretrained`, `disable_adapter()`).
- *Decode deliberately*: Section 5.

### When a check fails
| Error | Meaning | What to do |
|---|---|---|
| `Whole-ZIP SHA-256 mismatch` | the archive is not the one whose digest you were given | get the archive again; do not "fix" the expected hash |
| `SHA-256 mismatch: <file>` | a file inside the bundle differs from its manifest entry | the bundle was altered or corrupted in transit; reject it |
| `Manifest/file-set mismatch` | files were added to or removed from the bundle | reject it; a manifest that does not list a file cannot vouch for it |
| `Unsafe artifact path` / `Symlinks are not allowed` | the archive tries to write outside its folder | reject it; this is what a malicious archive looks like |
| `trustRemoteCode must be false` | the bundle asks to run model-repository code | reject it |
| `Invalid baseModelRevision provenance` | the bundle names a branch or tag instead of a commit | ask the producer to pin the revision |
| answers are English / off-style | wrong base revision, or a different tokenizer than the training one | check `provenance.json` against what was actually trained |

### Serving the adapter
- **Attach at load time**, as here, inside a Python service or batch job; the whole notebook is ~30 lines of load code.
- **Multi-adapter servers** (vLLM, SGLang, TGI) keep one base model resident and route requests to many adapters; give them `adapter_model.safetensors` and `adapter_config.json` from the bundle.
- **Merging** (`merge_and_unload()`) should be done on a 16-bit copy of the base model, not on the 4-bit one loaded here; merging into quantized weights loses precision. Merge, `save_pretrained`, then convert (e.g. to GGUF) for llama.cpp or Ollama.

### Licenses
`provenance.json` records the base model's license and the training dataset's license. Share-alike data (Dolly is CC-BY-SA) carries its terms into the adapter.
